In [ ]:
import pandas as pd
import muon as mu
import sys
import os



# Change path to wherever you have repo locally
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/cNMF_benchmarking/cNMF_benchmarking_pipeline')


from Plotting.src import rename_adata_gene_dictionary

from Evaluation.src import (
   compile_Program_loading_score_sheet_long, compile_Program_loading_score_sheet_flat, Compile_GO_sheet, \
    Compile_Geneset_sheet, Compile_Trait_sheet, Compile_Perturbation_sheet, Compile_Association_sheet, Compile_Explained_variance, \
    Compile_Target_Summary_sheet, Compile_Summary_sheet, load_simple_sheets, add_specificity_scores_file,check_program_name_match
)



In [ ]:
# respurse path
reference_gtf_path="/oak/stanford/groups/engreitz/Users/opushkar/genome/IGVFFI9573KOZR.gtf.gz"

# IO path
out_dir = '/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result'
run_name = '020326_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50'

# data path
mdata_path = '/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/020326_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu'


# save path for intermediate values
save_path = '/oak/stanford/groups/engreitz/Users/ymo/Project/combined_final_merged_hvg10k/Results/combined_final_merged_hvg10k/Eval/150_0_5'

# keys 
prog_key = 'cNMF'
data_key = 'rna'
categorical_key = 'timepoint'
guide_targets_key = "guide_targets"
num_gene = 300
components = [50]
sel_threshs = [0.2]
Sample = ['d0','d1','d2','d3']
perturbation_file_name = 'CRT'
non_tagerting_key = ['non-targeting']
effect_size = 'log2FC'

In [3]:
'''# rename genes if needed 
file_to_dictionary = "/oak/stanford/groups/engreitz/Users/ymo/Tools/cNMF_benchmarking/cNMF_benchmarking_pipeline/Evaluation/Resources/weissman_guides_with_coordinates.tsv"
result = rename_adata_gene_dictionary(mdata['rna'] ,dictionary_file_path=file_to_dictionary)
mdata.mod['rna'] = result'''

'# rename genes if needed \nfile_to_dictionary = "/oak/stanford/groups/engreitz/Users/ymo/Tools/cNMF_benchmarking/cNMF_benchmarking_pipeline/Evaluation/Resources/weissman_guides_with_coordinates.tsv"\nresult = rename_adata_gene_dictionary(mdata[\'rna\'] ,dictionary_file_path=file_to_dictionary)\nmdata.mod[\'rna\'] = result'

In [3]:
# reformat if needed 
mdata_guide_path = "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Data/withguide.h5ad"
mdata_guide = mu.read(mdata_guide_path)



In [4]:
# helper method 
def _assign_guide(mdata, mdata_guide):
        mdata['cNMF'].uns['guide_names'] = mdata_guide.uns['guide_names']
        mdata['cNMF'].uns['guide_targets'] = mdata_guide.uns['guide_targets']
        mdata['cNMF'].obsm['guide_assignment'] = mdata_guide.obsm['guide_assignment'].toarray()


In [ ]:
for sel_thresh in sel_threshs:
    for k in components:  

        output_folder = f"{out_dir}/{run_name}/Eval/{k}_{str(sel_thresh).replace('.','_')}"

        os.makedirs(output_folder, exist_ok=True)

        # Load mdata
        mdata = mu.read('{out_dir}/{run_name}/adata/cNMF_{k}_{sel_thresh}.h5mu'.format(out_dir = out_dir,
                                                                                run_name =run_name,
                                                                                k=k,
                                                                                sel_thresh = str(sel_thresh).replace('.','_')))                                                                      
        # assign guide
        _assign_guide(mdata)                                                    
          
 
        # load simple sheets
        df_Program_loading_long,df_Program_loading_flat, df_GO, df_Geneset, \
        df_Trait, df_Perturbation, df_Association, df_Explained_Variance, df_Perturbation_significant_gene_only = load_simple_sheets( 
            mdata, out_dir, run_name, k, sel_thresh, num_gene = num_gene,  Sample = Sample, perturbation_file_name = perturbation_file_name)

        # check program names match
        check_program_name_match(mdata, prog_key=prog_key, dataframes =[df_GO, df_Geneset, \
        df_Trait, df_Perturbation, df_Association, df_Explained_Variance, df_Perturbation_significant_gene_only ])


 
        # load target summary
        Perturbation_path_base = f'{out_dir}/{run_name}/Eval/{k}_{str(sel_thresh).replace(".", "_")}/{k}_{perturbation_file_name}'
        df_Target_Summary = Compile_Target_Summary_sheet(mdata, Perturbation_path_base, Sample = Sample, categorical_key = categorical_key, 
        prog_key = prog_key, data_key = data_key, guide_targets_key = guide_targets_key ,save_path=save_path, effect_size=effect_size)
        
        # load summary    
        df_Summary = Compile_Summary_sheet(mdata, df_GO, df_Geneset, df_Perturbation, df_Program_loading_flat, df_Explained_Variance, Sample = Sample, specicicity_path = save_path,
        categorical_key = categorical_key, non_tagerting_key=non_tagerting_key, effect_size=effect_size)

        # compling everything togehter
        print('Compliing Excel')
        MAX_ROWS = 1048575

        with pd.ExcelWriter(f'{output_folder}/cNMF_{k}_{str(sel_thresh).replace(".", "_")}.xlsx') as writer:
            df_Summary.to_excel(writer, sheet_name='Summary', index=True)
            df_Program_loading_long.to_excel(writer, sheet_name='Program Loadings', index=True)
            df_Target_Summary.to_excel(writer, sheet_name='Targets Summary', index=True)

            if df_Association is not None:
                df_Association.to_excel(writer, sheet_name='Sample Association', index=True)

            if df_Perturbation is not None:

                # update the df_Perturbation
                combined_conditions = []
                for samp in Sample:
                    df_Perturbation_ =  add_specificity_scores_file(save_path, Perturbation_path_base, samp)
                    df_Perturbation_['Sample'] = samp
                    combined_conditions.append(df_Perturbation_)

                df_Perturbation = pd.concat(combined_conditions)
                
                # separate the length 
                for i in range(0, len(df_Perturbation), MAX_ROWS):
                    sheet_num = i // MAX_ROWS + 1
                    df_Perturbation.iloc[i:i+MAX_ROWS].to_excel(writer, sheet_name=f'Perturbation Association {sheet_num}', index=True)
                for i in range(0, len(df_Perturbation), MAX_ROWS):
                    sheet_num = i // MAX_ROWS + 1
                    df_Perturbation_significant_gene_only.iloc[i:i+MAX_ROWS].to_excel(writer, sheet_name=f'significant regulators only {sheet_num}', index=True)


            if df_Trait is not None:
                df_Trait.to_excel(writer, sheet_name='Trait Enrichment', index=True)

            if df_GO is not None:
                df_GO.to_excel(writer, sheet_name='GO Term Enrichment', index=True)

            if df_Geneset is not None:
                df_Geneset.to_excel(writer, sheet_name='Geneset Enrichment', index=True)

In [ ]:
df_Perturbation_significant_gene_only